# Run Trellis 2-4B with Authentication on Modal

This notebook deploys the Trellis 2-4B image-to-3D model and queries its secure web endpoint with proxy authentication.

### Step 1: Deploy the Model

In [ ]:
!uv run modal deploy ../llm-hosting/trellis-2-4b.py

### Step 2: Load Credentials and Setup Endpoint

In [1]:
import os
import requests
import base64

def load_dotenv():
    for path in [".env", "../.env", "../../.env"]:
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        v = v.strip().strip("'\"")
                        os.environ.setdefault(k.strip(), v)
            break

load_dotenv()
MODAL_KEY = os.environ.get("MODAL_KEY")
MODAL_SECRET = os.environ.get("MODAL_SECRET")

if not MODAL_KEY or not MODAL_SECRET:
    print("WARNING: Credentials not loaded!")
else:
    print("Credentials loaded successfully!")

username = "sshibinthomass"
ENDPOINT_URL = f"https://{username}--trellis-2-4b-trellis2model-generate-api.modal.run"
print(f"Target Endpoint URL: {ENDPOINT_URL}")

Credentials loaded successfully!
Target Endpoint URL: https://sshibinthomass--trellis-2-4b-trellis2model-generate-api.modal.run


### Step 3: Run Authenticated Inference

In [4]:
from pathlib import Path
from datetime import datetime

image_path = "../llm-inference/img.png"
glb_bytes = test_inference(image_path)

if glb_bytes:
    os.makedirs("outputs", exist_ok=True)
    
    # Format output filename with input file name and current datetime
    input_stem = Path(image_path).stem
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"outputs/{input_stem}_{timestamp}.glb"
    
    with open(output_filename, "wb") as f:
        f.write(glb_bytes)
    print(f"Success! Generated GLB saved to {output_filename} ({len(glb_bytes)} bytes)")

Sending request to Trellis 2-4B secure endpoint...
Success! Generated GLB saved to outputs/img_20260628_122547.glb (4622448 bytes)


### Step 4: Verify Security (Unauthorized Calls)

In [ ]:
print("--- Test: Querying WITHOUT credentials ---")
response = requests.post(ENDPOINT_URL, json={"image_base64": "", "seed": 42})
print(f"Status: {response.status_code} (Expected: 401)")
print(f"Message: {response.text}")